# First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

# cell 1: ดึงข้อมูลปัจจุบัน (สัปดาห์ที่ 3 จะเขียนเป็น script จริงจัง สัปดาห์นี้ดูอย่างเดียว)

In [20]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()     # ดูตารางแก้ปัญหา: SSLError

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

# cell 2: ดู key ก่อนเสมอ อย่าเชื่อเอกสารนี้

In [21]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]          # ถ้า key ไม่ชื่อนี้ ให้แก้ตามที่เห็นจาก cell ก่อนหน้า
print(len(stations))
stations[0]   

<class 'dict'> ['stations']
180


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-22',
  'time': '20:00',
  'PM25': {'color_id': '1', 'aqi': '10', 'value': '5.8'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '2', 'aqi': '28', 'value': '37'},
  'CO': {'color_id': '1', 'aqi': '3', 'value': '0.60'},
  'NO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '2', 'aqi': '28', 'param': 'O3'}}}

# cell 3: แปลงเป็นตารางแบน

In [22]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()  

(180, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

ตอบคำถาม

1.หนึ่งแถวของ df_now คืออะไร (หนึ่งสถานี ณ เวลาล่าสุด?)
:แถวที่เห็นคือ 119t สวนสาธารณะธารา อยู่ที่ กระบี่ วันที่ 2026-11-22 เวลาล่าสุดคือ 20:00

2.คอลัมน์ใดเก็บรหัสสถานี ชื่อ พิกัด วันที่ และเวลา
:รหัสสถานี คือ stationID
ชื่อ คือ nameTH,nameEN
พิกัด คือ lat,long
วันที่ คือ AQILast
เวลา คือ time

3.ค่า PM2.5 อยู่คอลัมน์ไหน มีทั้ง ค่าความเข้มข้น และ ค่า AQI หรือไม่ หน่วยคืออะไร
:ค่าpm2.5 อยู่คอลัมน์ 'PM25' มีค่าความเข้มข้น 5.8 และ ค่า aqi 10 ไม่ได้มีหน่วยบอกแต่หน่วยวัดตามมาตรฐานคือ μg/m³

4.ค่าตัวเลขถูกเก็บเป็นข้อความ (object) หรือตัวเลข มีค่าแปลก เช่น ค่าลบ หรือ "-" แทนค่าหายหรือไม่
: ค่าตัวเลขถูกเก็บเป็น ข้อความ มีค่าแปลกคือจากคอลัมน์ PM10 มีค่า value เป็น -1 และมี aqi -1

# cell 4: ตรวจค่าหายและค่าแปลก (แทน <pm25_col> ด้วยชื่อคอลัมน์จริง)


In [24]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 1


count    180.000000
mean      14.938333
std        5.300071
min       -1.000000
25%       11.900000
50%       14.950000
75%       17.725000
max       35.900000
Name: AQILast.PM25.value, dtype: float64

# ขั้นที่ 5: first look ไฟล์ตัวอย่างย้อนหลัง (เฉพาะช่วงสำรวจ)
ถ้าดึงข้อมูลในขั้นที่ 4 ไม่ได้ ให้ข้ามมาขั้นนี้ได้เลย

In [ ]:
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ใช้เฉพาะช่วงสำรวจ 2563–2566 (ค.ศ. 2020–2023) ห้ามดูช่วงทดสอบ
explore = hourly[hourly["ts"] < "2024-01-01"]

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts"].max())
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(0, 8) 0 สถานี
NaT → NaT
Series([], Name: count, dtype: int64)


station_id   NaN
ts           NaN
pm25         NaN
pm10         NaN
o3           NaN
co           NaN
no2          NaN
so2          NaN
dtype: float64

In [15]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

Series([], Name: pm25, dtype: float64)

# สรุปสิ่งที่เห็น 3–5 ข้อ

In [19]:
print(hourly.shape)                        # (จำนวนแถว, จำนวนคอลัมน์)
print(hourly.dtypes)                         # ts เป็น datetime หรือยัง
print(hourly["ts"].agg(["min", "max"]))    # ช่วงเวลาของข้อมูล
print(hourly["station_id"].nunique())    # จำนวนสถานี
print(hourly.isna().mean().round(3))        # สัดส่วนค่าหายต่อคอลัมน์
print(hourly["pm25"].describe())            # ค่าติดลบหรือสูงผิดปกติไหม


(333216, 8)
station_id            object
ts            datetime64[ns]
pm25                 float64
pm10                 float64
o3                   float64
co                   float64
no2                  float64
so2                  float64
dtype: object
min   2026-09-01 00:00:00
max   2026-11-17 23:00:00
Name: ts, dtype: datetime64[ns]
178
station_id    0.000
ts            0.000
pm25          0.071
pm10          0.646
o3            0.849
co            0.811
no2           0.794
so2           0.828
dtype: float64
count    309435.000000
mean         15.241451
std           8.708748
min           0.000000
25%           9.100000
50%          13.600000
75%          19.500000
max         167.900000
Name: pm25, dtype: float64


1 พบว่า จากไฟล์ pm2.5 hourly มี 333216 แถว , 8 คอลัมน์

2 พบว่า จากไฟล์ pm2.5 hourly คอลัมน์ PM25และPM10 มีการเก็บค่าตัวเลขเป็นข้อความและ ในส่วนของ PM2.5 aqi: 9 และ value : 5.1 ส่วนของ PM10 พบค่าแปลก aqi : -1 , value : -1

3 พบว่า จากไฟล์ pm2.5 hourly station_id มี 178 สถานี

4 พบว่า จากไฟล์ในขั้นที่5 มีค่าว่างเนื่องจากในไฟล์ตัวอย่างย้อนหลังเป็นไฟล์ปี 2569 แต่ในเงื่อนไขสำรวจปีที่น้อยกว่า 2566 

In [ ]:
# # df = df_now['AQILast.PM25.value']
# # df.to_csv('ขอเช็คหน่อยยยยย')
# df = df_now['AQILast.PM10.value']
# df.to_csv('เช็คpm10')
